# 12c · 蒸馏，以及浏览器里的代价

**前几本的结论。**最准的是"图像 + 笔迹"的方案：notebook 11 的**晚期融合**（2 个网络）、notebook 12b 的**交叉注意力 `ink_glance`**（1 个网络，24 万参数），以及 notebook 12 发现的**更好的图像模型（resnet、separable）+ 轨迹模型**（手指 99.51%、99.44%）。另外，多个模型平均（集成）通常还能再好一点，但上线要跑好几个模型。

**这一本回答两个问题：**

1. **蒸馏 [23]**：能不能把"好几个模型的平均"教给**一个**模型，甚至是更小的模型？
   - **图像蒸馏**：老师 = notebook 12 里验证集最好的几个图像模型的集成；学生 = 上线的小 CNN（10.7 万参数）。只用图像，作为"不加笔迹"时的最佳选择。
   - **联合蒸馏**：老师 = `ink_glance` 两个种子 + 晚期融合两个种子的平均（没有笔迹的训练样本用 `ink_glance`（不给笔迹）和图像模型 C 代替晚期融合）；学生 = 同样大小的 `ink_glance`，以及一个**缩小版**（通道减半，约 6 万参数）。
   - 对照：缩小版**不蒸馏**直接训练，用来分清"变小"和"蒸馏"各自的影响。
2. **浏览器代价**：所有候选导出为 TF.js（float16 权重，和上线的 `digit_model` 相同），用游戏的 WASM 后端测加载时间和单次推理延迟，并检查浏览器的输出和 Python 一致。

**蒸馏的做法。**学生的损失 = 0.5 × 普通交叉熵（label smoothing 0.1，和以前一样）+ 0.5 × $T^2$ × KL(老师 ‖ 学生)，温度 $T = 2$ [23]。老师在每个训练批次上**实时**计算（看到的是和学生同样增强过的输入）。训练数据、预算和 12b 完全相同，每个学生两个种子。

**一个上线障碍和它的解决。**Keras 的注意力层用到 `Einsum` 运算，TF.js 的 WASM 后端没有实现它（实测报错 `Kernel 'Einsum' not registered for backend 'wasm'`）。`archs.CrossAttention` 用普通矩阵乘法重写了同一个计算，`archs.wasm_ink_glance` 把训练好的权重原样搬过去；下面第 1 节验证两者输出相同。

In [1]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

sys.path.insert(0, '.')
import archs, datasets, evaluate, models, multiview, recipe, ship, tfjs_export

REAL = Path('cache/real'); RESULTS = Path('results'); MODEL_DIR = Path('cache/models')
EXPORT = Path('cache/tfjs12c'); EXPORT.mkdir(parents=True, exist_ok=True)
SEEDS = recipe.SEEDS
T, ALPHA = 2.0, 0.5
plt.rcParams['figure.dpi'] = 110

meta, x_real = recipe.real_meta()
y = meta.label.to_numpy()
SPLITS = recipe.eval_splits(meta)
V = dict(np.load(REAL / 'views12b.npz'))
real_om, real_tr = V['real_om'].astype(np.float32), V['real_tr'].astype(np.float32)
JK = archs.JOINT_INPUTS['ink_glance']
real_in = lambda avail=1.0: multiview.eval_inputs(x_real, real_om, real_tr, JK, avail)
acc = lambda p, idx: 100 * float((p[idx].argmax(1) == y[idx]).mean())

## 1. 注意力层改写后输出不变

In [2]:
ig = {s: keras.models.load_model(MODEL_DIR / f'joint12b_ink_glance_s{s}.keras') for s in SEEDS}
ig_wasm = {s: archs.wasm_ink_glance(m) for s, m in ig.items()}
probe = multiview.eval_inputs(x_real[:512], real_om[:512], real_tr[:512], JK)
for s in SEEDS:
    print(f'seed {s}: max |p(MultiHeadAttention) - p(CrossAttention)| =',
          float(np.abs(ig[s].predict(probe, verbose=0) - ig_wasm[s].predict(probe, verbose=0)).max()))

seed 42: max |p(MultiHeadAttention) - p(CrossAttention)| = 1.1920928955078125e-07


seed 43: max |p(MultiHeadAttention) - p(CrossAttention)| = 1.1920928955078125e-07


## 2. 老师

**图像老师**：notebook 12 里按验证集（触控笔 + 手指的平均）排名前两位的结构，各两个种子，共 4 个模型。

In [3]:
r12 = json.loads((RESULTS / '12_image_architectures.json').read_text(encoding='utf-8'))
rank = pd.Series({a: v['val mean'] for a, v in r12['mean'].items()}).sort_values(ascending=False)
print('image architectures by validation accuracy:'); print(rank.round(3))
TOP = list(rank.index[:2])
def img_path(a, s):
    return MODEL_DIR / (f'real09_C_s{s}.keras' if a == 'cnn' else f'arch12_{a}_s{s}.keras')
img_teachers = [keras.models.load_model(img_path(a, s)) for a in TOP for s in SEEDS]
img_C = [keras.models.load_model(img_path('cnn', s)) for s in SEEDS]
r11 = json.loads((RESULTS / '11_trajectory_and_fusion.json').read_text(encoding='utf-8'))
W11 = r11['fusion_weight']; traj_name = r11['trajectory_chosen'].replace(' + ', '_')
trajs = [keras.models.load_model(MODEL_DIR / f'traj11_{traj_name}_s{s}.keras') for s in SEEDS]
print('image teacher:', TOP, '| late fusion weight', W11)

image architectures by validation accuracy:
resnet       99.469
separable    99.324
cnn_wide     99.312
convmixer    99.184
cnn          99.172
dtype: float64


image teacher: ['resnet', 'separable'] | late fusion weight 0.55


In [4]:
def image_teacher(x):
    return tf.add_n([m(x, training=False) for m in img_teachers]) / len(img_teachers)

def joint_teacher(d):
    """ink_glance x2 and late fusion x2 where the pen path is known; ink_glance x2 (no path) and image C x2 where not."""
    ig_p = tf.add_n([m(d, training=False) for m in ig.values()]) / 2
    pc = [m(d['img'], training=False) for m in img_C]
    lf = [tf.nn.softmax(W11 * tf.math.log(p + 1e-7) + (1 - W11) * tf.math.log(t(d['traj'], training=False) + 1e-7))
          for p, t in zip(pc, trajs)]
    with_path = (ig_p + (lf[0] + lf[1]) / 2) / 2
    without = (ig_p + (pc[0] + pc[1]) / 2) / 2
    return tf.where(d['avail'] > 0.5, with_path, without)

# the teachers on the real data (for reference in the results table)
P = {}
P['teacher: image ensemble'] = np.concatenate([image_teacher(tf.constant(x_real[i:i + 4096, ..., None])).numpy()
                                               for i in range(0, len(y), 4096)])
P['teacher: joint ensemble'] = np.concatenate([joint_teacher({k: tf.constant(v[i:i + 4096]) for k, v in real_in().items()}).numpy()
                                               for i in range(0, len(y), 4096)])
for k in ['teacher: image ensemble', 'teacher: joint ensemble']:
    print(k, {s: round(acc(P[k], idx), 2) for s, idx in SPLITS.items()})

teacher: image ensemble {'val: pen': 99.53, 'val: finger': 99.36, 'test: pen': 98.2, 'test: finger': 99.27}
teacher: joint ensemble {'val: pen': 99.73, 'val: finger': 99.31, 'test: pen': 99.2, 'test: finger': 99.38}


## 3. 蒸馏训练

`Distiller` 包住学生：每一步先让老师算出概率，再按上面的损失更新学生。学生存在 `cache/models/kd12c_<名字>_s<种子>.keras`。

In [5]:
class Distiller(keras.Model):
    def __init__(self, student, teacher, alpha=ALPHA, temperature=T):
        super().__init__()
        self.student, self.teacher, self.alpha, self.t = student, teacher, alpha, temperature
        self.ce = keras.losses.CategoricalCrossentropy(label_smoothing=recipe.SMOOTHING)

    def call(self, x, training=False):
        return self.student(x, training=training)

    def train_step(self, data):
        x, yb = data
        pt = self.teacher(x)
        soft_t = tf.nn.softmax(tf.math.log(pt + 1e-7) / self.t)
        with tf.GradientTape() as tape:
            ps = self.student(x, training=True)
            soft_s = tf.nn.log_softmax(tf.math.log(ps + 1e-7) / self.t)
            kd = tf.reduce_mean(tf.reduce_sum(soft_t * (tf.math.log(soft_t + 1e-7) - soft_s), -1))
            loss = self.alpha * self.ce(yb, ps) + (1 - self.alpha) * self.t ** 2 * kd
        g = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(g, self.student.trainable_variables))
        return {'loss': loss}

x_mn, y_mn, _, _ = datasets.load_mnist(); x_mn = x_mn.astype(np.float32) / 255
skel, y_skel = ship.skeleton_sets(None, None, 'skel_train', seed=1, pens=True)
IMAGE_SOURCES = [(x_mn, y_mn, models.augment_mnist, 0.35), (skel['area18'], y_skel, models.augment_light, 0.35)]
PAIRED = [(V['pen_img'], V['pen_om'], V['pen_tr'], V['pen_y'], 0.2), (V['tmpl_img'], V['tmpl_om'], V['tmpl_tr'], V['tmpl_y'], 0.1)]
src_c = recipe.sources_c(meta, recipe.strokes())

SMALL = dict(w=16, tw=(24, 48))
STUDENTS = {   # name: (builder, data, teacher or None)
    'cnn KD (image teacher)': (lambda: models.build_cnn('cnn_kd'), 'image', image_teacher),
    'ink_glance KD': (lambda: archs.ink_glance('ink_glance_kd'), 'joint', joint_teacher),
    'ink_glance small KD': (lambda: archs.ink_glance('ink_glance_small_kd', **SMALL), 'joint', joint_teacher),
    'ink_glance small (no KD)': (lambda: archs.ink_glance('ink_glance_small', **SMALL), 'joint', None),
}

def slug(name):
    return name.replace(' (image teacher)', '').replace(' (no KD)', '_plain').replace(' ', '_')

def train_student(name, seed):
    path = MODEL_DIR / f'kd12c_{slug(name)}_s{seed}.keras'
    if path.exists():
        return keras.models.load_model(path), None
    build, data, teacher = STUDENTS[name]
    keras.utils.set_random_seed(seed)
    if data == 'image':
        ds = models.make_train_ds(src_c, batch=recipe.BATCH, seed=seed).map(lambda a, b: (a, tf.one_hot(b, 10)))
    else:
        ds = multiview.joint_ds(IMAGE_SOURCES, PAIRED, JK, batch=recipe.BATCH, seed=seed)
    student = build()
    t0 = time.time()
    if teacher is None:
        recipe.compile_and_fit(student, ds, seed)
    else:
        d = Distiller(student, teacher)
        d.compile(optimizer=keras.optimizers.Adam(keras.optimizers.schedules.CosineDecay(recipe.LR, recipe.STEPS * recipe.EPOCHS)))
        d.fit(ds, steps_per_epoch=recipe.STEPS, epochs=recipe.EPOCHS, verbose=0)
    student.save(path)
    return student, time.time() - t0

students, rows = {}, []
for name, (_, data, _) in STUDENTS.items():
    for seed in SEEDS:
        m, secs = train_student(name, seed)
        students[(name, seed)] = m
        P[(name, seed)] = m.predict(x_real[..., None] if data == 'image' else real_in(), batch_size=1024, verbose=0)
        r = {'model': name, 'seed': seed, 'params': m.count_params(), 'train min': None if secs is None else round(secs / 60, 1)}
        r.update({k: acc(P[(name, seed)], idx) for k, idx in SPLITS.items()})
        rows.append(r); print(name, seed, {k: round(v, 2) for k, v in r.items() if isinstance(v, float)})

cnn KD (image teacher) 42 {'val: pen': 99.53, 'val: finger': 98.83, 'test: pen': 98.46, 'test: finger': 98.74}


cnn KD (image teacher) 43 {'val: pen': 99.4, 'val: finger': 98.86, 'test: pen': 98.31, 'test: finger': 98.73}


ink_glance KD 42 {'val: pen': 99.67, 'val: finger': 99.29, 'test: pen': 99.03, 'test: finger': 99.34}


ink_glance KD 43 {'val: pen': 99.73, 'val: finger': 99.21, 'test: pen': 99.14, 'test: finger': 99.37}


ink_glance small KD 42 {'val: pen': 99.8, 'val: finger': 98.94, 'test: pen': 99.0, 'test: finger': 99.1}


ink_glance small KD 43 {'val: pen': 99.67, 'val: finger': 99.16, 'test: pen': 99.09, 'test: finger': 99.22}


ink_glance small (no KD) 42 {'val: pen': 99.67, 'val: finger': 99.06, 'test: pen': 99.37, 'test: finger': 98.98}


ink_glance small (no KD) 43 {'val: pen': 99.6, 'val: finger': 98.99, 'test: pen': 99.31, 'test: finger': 98.87}


## 4. 准确率：学生、老师和以前的方案

两个种子的平均。`networks` = 上线时要跑的网络个数。

In [6]:
P9 = dict(np.load(REAL / 'probs09.npz')); P12 = dict(np.load(REAL / 'probs12.npz'))
P12b = dict(np.load(REAL / 'probs12b.npz'))
ref = {'image C (09, shipped recipe)': [P9[f'C_{s}'] for s in SEEDS],
       f'best image arch ({TOP[0]})': [P12[f'{TOP[0]}_{s}'] for s in SEEDS],
       'late fusion (11)': [P12b[f'late fusion (11)|{s}'] for s in SEEDS],
       'ink_glance (12b)': [P12b[f'ink_glance|{s}'] for s in SEEDS]}

def fuse(p_img, p_traj, w):
    z = w * np.log(p_img + 1e-7) + (1 - w) * np.log(p_traj + 1e-7)
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

W_LATE = {'late fusion (11)': W11}
for a in ['resnet', 'separable']:                          # notebook 12: better image models + the same trajectory model
    W_LATE[f'{a} + trajectory'] = r12['fusion_with_trajectory'][a]['w']
    ref[f'{a} + trajectory'] = [fuse(P12[f'{a}_{s}'], P12b[f'trajectory (11)|{s}'], W_LATE[f'{a} + trajectory']) for s in SEEDS]
for nm, ps in ref.items():
    for s, p in zip(SEEDS, ps):
        P[(nm, s)] = p
        rows.append({'model': nm, 'seed': s, **{k: acc(p, idx) for k, idx in SPLITS.items()}})
for nm in ['teacher: image ensemble', 'teacher: joint ensemble']:
    rows.append({'model': nm, 'seed': 0, **{k: acc(P[nm], idx) for k, idx in SPLITS.items()}})
runs = pd.DataFrame(rows)
cols = list(SPLITS)
mean = runs.groupby('model', sort=False)[cols].mean()
mean['val mean'] = mean[['val: pen', 'val: finger']].mean(axis=1)
NETS = {'late fusion (11)': 2, 'resnet + trajectory': 2, 'separable + trajectory': 2, 'teacher: image ensemble': 4, 'teacher: joint ensemble': 6}
mean['networks'] = [NETS.get(m, 1) for m in mean.index]
mean.round(2)

,val: pen,val: finger,test: pen,test: finger,val mean,networks
model,,,,,,
cnn KD (image teacher),99.47,98.84,98.38,98.73,99.15,1
ink_glance KD,99.70,99.25,99.09,99.35,99.47,1
ink_glance small KD,99.73,99.05,99.04,99.16,99.39,1
ink_glance small (no KD),99.63,99.03,99.34,98.92,99.33,1
"image C (09, shipped recipe)",99.57,98.78,98.17,98.78,99.17,1
best image arch (resnet),99.53,99.41,98.26,99.29,99.47,1
late fusion (11),99.73,99.16,99.00,99.23,99.45,2
ink_glance (12b),99.67,99.14,99.04,99.17,99.40,1
resnet + trajectory,99.70,99.46,98.81,99.51,99.58,2


## 5. 在游戏规则下（屏幕上 3 个答案，种子 42）

In [7]:
CANDIDATES = ['image C (09, shipped recipe)', f'best image arch ({TOP[0]})', 'cnn KD (image teacher)',
              'late fusion (11)', 'resnet + trajectory', 'separable + trajectory',
              'ink_glance (12b)', 'ink_glance KD', 'ink_glance small KD', 'ink_glance small (no KD)']
game = []
for k in ['test: pen', 'test: finger']:
    idx = SPLITS[k]
    for nm in CANDIDATES:
        a_, w_, o_ = evaluate.game_hit_rate(P[(nm, 42)][idx], y[idx], n_answers=3, seed=8)
        game.append({'test set': k, 'model': nm, 'accepted %': 100 * a_, 'wrong yokai %': 100 * w_, 'rewrite / miss %': 100 * o_})
game = pd.DataFrame(game)
game.round(2)

,test set,model,accepted %,wrong yokai %,rewrite / miss %
0,test: pen,"image C (09, shipped recipe)",98.71,0.43,0.86
1,test: pen,best image arch (resnet),98.34,0.49,1.17
2,test: pen,cnn KD (image teacher),98.68,0.37,0.94
3,test: pen,late fusion (11),99.23,0.23,0.54
4,test: pen,resnet + trajectory,98.97,0.29,0.74
5,test: pen,separable + trajectory,99.17,0.20,0.63
6,test: pen,ink_glance (12b),99.09,0.29,0.63
7,test: pen,ink_glance KD,99.40,0.17,0.43
8,test: pen,ink_glance small KD,99.26,0.26,0.49
9,test: pen,ink_glance small (no KD),99.46,0.11,0.43


## 6. 导出到浏览器

每个候选（种子 42）导出为 TF.js 图模型（float16 权重），放在 `cache/tfjs12c/`（不进 git）。同时导出 32 个真实测试样本的输入和 Python 的输出，浏览器端用来核对。晚期融合是两个文件（图像 + 轨迹），在浏览器里分别运行再按各自的 $w$ 融合。

然后在浏览器里打开 `export/bench12c.html`（需要从仓库根目录起一个静态服务器），它会用 **WASM 后端**（游戏用的）对每个模型测：加载时间、单个样本推理延迟（200 次的中位数）、和 Python 输出的差别。结果存成 `results/12c_browser.json`，再运行本 notebook 时下一节会读入。

In [8]:
EXPORTS = {   # name -> list of (file stem, keras model, input kind)
    'image C (09, shipped recipe)': [('image_C', img_C[0], 'img')],
    f'best image arch ({TOP[0]})': [('best_image', keras.models.load_model(img_path(TOP[0], 42)), 'img')],
    'cnn KD (image teacher)': [('cnn_kd', students[('cnn KD (image teacher)', 42)], 'img')],
    'late fusion (11)': [('image_C', img_C[0], 'img'), ('traj', trajs[0], 'traj')],
    'resnet + trajectory': [('resnet', keras.models.load_model(img_path('resnet', 42)), 'img'), ('traj', trajs[0], 'traj')],
    'separable + trajectory': [('separable', keras.models.load_model(img_path('separable', 42)), 'img'), ('traj', trajs[0], 'traj')],
    'ink_glance (12b)': [('ink_glance', ig_wasm[42], 'joint')],
    'ink_glance KD': [('ink_glance_kd', archs.wasm_ink_glance(students[('ink_glance KD', 42)]), 'joint')],
    'ink_glance small KD': [('ink_glance_small_kd', archs.wasm_ink_glance(students[('ink_glance small KD', 42)], **SMALL), 'joint')],
    'ink_glance small (no KD)': [('ink_glance_small', archs.wasm_ink_glance(students[('ink_glance small (no KD)', 42)], **SMALL), 'joint')],
}
sizes, done = {}, {}
for nm, parts in EXPORTS.items():
    kb = 0
    for stem, m, kind in parts:
        if stem not in done:
            done[stem] = tfjs_export.export(m, EXPORT / stem, Path('cache') / f'saved12c_{stem}')
        kb += done[stem]
    sizes[nm] = kb

idx = SPLITS['test: finger'][:16].tolist() + SPLITS['test: pen'][:16].tolist()
check = {'img': np.round(x_real[idx], 4).tolist(), 'traj': np.round(real_tr[idx], 5).tolist(), 'y': y[idx].tolist(), 'models': []}
for nm, parts in EXPORTS.items():
    p = P[(nm, 42)][idx]
    check['models'].append({'name': nm, 'parts': [{'url': f'../cache/tfjs12c/{stem}/model.json', 'kind': kind} for stem, _, kind in parts],
                            'w': W_LATE.get(nm), 'probs': np.round(p, 5).tolist(), 'kb': sizes[nm]})
(EXPORT / 'check.json').write_text(json.dumps(check))
pd.Series(sizes, name='TF.js size KB (float16)')

INFO:tensorflow:Assets written to: cache\saved12c_image_C\assets


INFO:tensorflow:Assets written to: cache\saved12c_image_C\assets


INFO:tensorflow:Assets written to: cache\saved12c_best_image\assets


INFO:tensorflow:Assets written to: cache\saved12c_best_image\assets


INFO:tensorflow:Assets written to: cache\saved12c_cnn_kd\assets


...

INFO:tensorflow:Assets written to: cache\saved12c_ink_glance_small_kd\assets


INFO:tensorflow:Assets written to: cache\saved12c_ink_glance_small_kd\assets


INFO:tensorflow:Assets written to: cache\saved12c_ink_glance_small\assets


INFO:tensorflow:Assets written to: cache\saved12c_ink_glance_small\assets


image C (09, shipped recipe)    222
best image arch (resnet)        376
cnn KD (image teacher)          222
late fusion (11)                471
resnet + trajectory             625
separable + trajectory          373
ink_glance (12b)                578
ink_glance KD                   581
ink_glance small KD             234
ink_glance small (no KD)        232
Name: TF.js size KB (float16), dtype: int64

## 7. 浏览器结果和最终比较

In [9]:
BROWSER = RESULTS / '12c_browser.json'
final = mean.loc[CANDIDATES, ['test: pen', 'test: finger', 'networks']].copy()
final['size KB'] = pd.Series(sizes)
g = game.pivot(index='model', columns='test set', values='wrong yokai %')
final['wrong yokai % (finger)'] = g['test: finger']; final['wrong yokai % (pen)'] = g['test: pen']
if BROWSER.exists():
    b = json.loads(BROWSER.read_text(encoding='utf-8'))
    final['WASM ms / digit'] = pd.Series({m['name']: m['predictMs'] for m in b['models']})
    final['load ms'] = pd.Series({m['name']: m['loadMs'] for m in b['models']})
    final['same answer as Python (of 32)'] = pd.Series({m['name']: m['argmaxAgree'] for m in b['models']})
    print('browser:', b['userAgent'], '| backend', b['backend'])
else:
    print('no browser results yet: open export/bench12c.html, save results/12c_browser.json, run again')
final.round(2)

browser: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Claude/3.6556.2 Chrome/152.0.7977.167 Safari/537.36 MSIX | backend wasm


,test: pen,test: finger,networks,size KB,wrong yokai % (finger),wrong yokai % (pen),WASM ms / digit,load ms,same answer as Python (of 32)
model,,,,,,,,,
"image C (09, shipped recipe)",98.17,98.78,1,222,0.29,0.43,3.40,332,32
best image arch (resnet),98.26,99.29,1,376,0.17,0.49,5.10,343,32
cnn KD (image teacher),98.38,98.73,1,222,0.31,0.37,2.45,335,32
late fusion (11),99.00,99.23,2,471,0.16,0.23,3.75,331,32
resnet + trajectory,98.81,99.51,2,625,0.10,0.29,5.55,343,32
separable + trajectory,98.99,99.44,2,373,0.11,0.20,3.10,338,32
ink_glance (12b),99.04,99.17,1,578,0.20,0.29,5.80,335,32
ink_glance KD,99.09,99.35,1,581,0.19,0.17,5.45,342,32
ink_glance small KD,99.04,99.16,1,234,0.21,0.26,3.40,342,32


## 8. 保存

In [10]:
out = {'image_teacher': TOP, 'runs': runs.to_dict(orient='records'), 'mean': mean.round(3).to_dict(orient='index'),
       'game_3_answers': game.round(3).to_dict(orient='records'), 'final': final.round(3).reset_index().to_dict(orient='records'),
       'kd': {'T': T, 'alpha': ALPHA}, 'small': {'w': SMALL['w'], 'tw': list(SMALL['tw'])}, 'seeds': SEEDS}
(RESULTS / '12c_distill_and_browser.json').write_text(json.dumps(out, indent=1, default=float), encoding='utf-8')
print('saved results/12c_distill_and_browser.json')

saved results/12c_distill_and_browser.json


## 9. 结论

**1. 浏览器里所有候选都很快，速度不是选择的约束。**WASM 后端（游戏用的）上，单个数字的推理时间都在 2.5–6 ms，两个网络的融合也只要 3–6 ms，远小于一帧（16 ms），而且这只在玩家写完一笔时运行一次。两次完整测量之间相差可达 2 ms，所以 2 ms 以内的差别没有意义。加载时间都在 0.33–0.35 s，主要是固定开销，和模型大小关系不大。**所有候选在浏览器里的答案都和 Python 完全一致**（32/32，概率差 ≤ 2×10⁻⁴，来自 float16 权重）。注意力层改写后（`archs.CrossAttention`）输出和原来的差别是 10⁻⁷ 量级。

**2. 蒸馏对 ink_glance 有用，对上线大小的 CNN 没用。**

| 学生 | 老师 | 手指 | 触控笔 |
|---|---|---|---|
| cnn（10.7 万参数） | resnet + separable 的集成 | 98.78% → 98.73% | 98.17% → 98.38% |
| ink_glance（24.4 万） | ink_glance + 晚期融合 | 99.17% → **99.35%** | 99.04% → 99.09% |
| ink_glance 缩小版（6.4 万） | 同上 | 98.92% → **99.16%** | 99.34% → 99.04% |

- 图像 CNN 学不到 resnet 的优势：老师的好处来自结构（残差和 BatchNorm 让优化更好），而不是"软标签"里的信息，和 12 的观察（集成几乎不比单个 resnet 好）一致。
- ink_glance 学生几乎追上了 6 个网络的老师（手指 99.35% 对 99.38%）：一个网络、一个文件，就拿到了集成的效果。
- 缩小版只有 234 KB（和现在上线的 222 KB 一样大），蒸馏后手指 99.16%，比现在游戏里的模型（98.36%）错误少一半。触控笔上不蒸馏的版本反而更高（99.34% 对 99.04%），但触控笔测试集只有 14 人，这个差距不可信；验证集上两者都是 99.6–99.7%。

**3. 最终的候选**（测试集；按验证集排名，`val mean` 是触控笔和手指验证集的平均）：

| 方案 | 网络 | 大小 | WASM 延迟 | 手指 | 触控笔 | 验证集 | 写对时打错妖怪（手指） |
|---|---|---|---|---|---|---|---|
| **resnet + 轨迹** | 2 | 625 KB | 5.6 ms | **99.51%** | 98.81% | **99.58%** | **0.10%** |
| separable + 轨迹 | 2 | 373 KB | 3.1 ms | 99.44% | 98.99% | 99.48% | 0.11% |
| ink_glance KD | 1 | 581 KB | 5.5 ms | 99.35% | 99.09% | 99.47% | 0.19% |
| resnet（只看图像） | 1 | 376 KB | 5.1 ms | 99.29% | 98.26% | 99.47% | 0.17% |
| ink_glance 缩小版 KD | 1 | 234 KB | 3.4 ms | 99.16% | 99.04% | 99.39% | 0.21% |
| 现在的配方（09 C） | 1 | 222 KB | 3.4 ms | 98.78% | 98.17% | 99.17% | 0.29% |

**推荐 resnet + 轨迹**：验证集第一，手指测试集最高，打错妖怪最少；代价是 625 KB（比现在多 400 KB，float16）和两个模型文件，延迟仍然只有约 5 ms。**separable + 轨迹**是轻量的备选：只多 150 KB、更快，准确率几乎一样。如果只想维护一个文件，**ink_glance KD** 是最好的单网络方案，但需要用改写过的注意力层导出（本 notebook 已经做好）。